In [0]:
#Masterclass dados ausentes
from pyspark.sql.functions import *

In [0]:
data = [
    ("Ricardo", 45, "Salvador"),
    ("Fernanda", None, "Recife"),
    ("Gabriel", None, None),
    ("Leandro", 32, None),
    ("Lucas", 29, None)
]

columns = ["nome","idade","cidade"]

df_null = spark.createDataFrame(data = data, schema = columns)

In [0]:
#df = df_null.withColumn("Idade é nula ?", when(col("idade").isNull(), True).otherwise(False))
df = df_null.withColumn("Idade é nula ?", isnull(col("idade")))
display(df)

In [0]:
df_filter = df.filter(col("Idade é nula ?") == True)
display(df_filter)

In [0]:
df_fillna = df.fillna(0, subset=["idade"])
display(df_fillna)

In [0]:
df_dropna = df.dropna(subset=["idade"])
display(df_dropna)

In [0]:
#Dados ausentes / espaços ===== Como resolver ? (trim)
data = [
    ("Ricardo", 45, ""),
    ("Fernanda", None, "       Recife         "),
    ("Leonardo", " ", None),
    ("Mike", "", "Recife"),
    ("Kyel", 22, None)
]

columns = ["nome", "idade", "cidade"]

df_ausencia = spark.createDataFrame(data = data, schema = columns)

display(df_ausencia)

In [0]:
df_ausencia = df_ausencia.dropna(subset=["idade"])
df_ausencia = df_ausencia.withColumn("idade", trim(col("idade")))
df_ausencia = df_ausencia.withColumn("idade null", isnull(col("idade")))
display(df_ausencia)

In [0]:
df_ausencia = df_ausencia.withColumn("idade", when(col("idade") == "", lit(None)).otherwise(col("idade")))
display(df_ausencia)

In [0]:
media_idade = df_ausencia.agg(avg(col("idade"))).collect()[0][0]
df_ausencia = df_ausencia.withColumn("idade", col("idade").cast("double"))
df_ausencia = df_ausencia.fillna(media_idade, subset=["idade"])

display(df_ausencia)

In [0]:
#Dataframe com tabulação ---- \t -> tab | rlike (regular expression like)
data_tab = [
    ("João", "\t", "Salvador"),
    ("Pedro", "35\t", "Curitiba"),
    ("Lucas", "\t28", "Porto Alegre"),
    ("Julia", "40", "Florianópolis")
]

colunas_tab = ["nome", "idade", "cidade"]

df_tab = spark.createDataFrame(data_tab, colunas_tab)

display(df_tab)

In [0]:
import builtins

df_tab = df_tab.withColumn("idade", regexp_replace(col("idade"), r'\s+', '')) # r'\s+' -> tabs e espaço
df_tab = df_tab.withColumn("idade", when(col("idade") == "", lit(None)).otherwise(col("idade")))
media_df_tab_idade = df_tab.agg(avg(col("idade"))).collect()[0][0]
df_tab = df_tab.withColumn("idade", col("idade").cast("double"))
df_tab = df_tab.fillna(builtins.round(media_df_tab_idade), subset=["idade"])
display(df_tab)

#Se rodar df_tab = df_tab.withColumn("idade", regexp_replace(col("idade"), r'\s+', '')) \
                         #.withColumn("cidade", regexp_replace(col("cidade"), r'\s+', '')) vai pegar as duas colunas